# Day 49: TensorRT-LLM: Compile a Model and Compare

**Layer:** Implementation


## Concept Overview

TensorRT-LLM compiles LLMs to GPU-specific engines by selecting optimal kernels, fusing operations, and calibrating precision. The compile step is offline; inference is online.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Deploy TensorRT-LLM: Compile and Compare

TensorRT-LLM compiles a PyTorch model into a GPU-specific engine with operator fusion and INT8/FP8 calibration. Comparing against eager PyTorch quantifies the compilation benefit.


In [ ]:
print('TensorRT-LLM compilation pipeline:')
print()
print('Step 1: Convert model to TRT-LLM format')
print('  python convert_checkpoint.py \\')
print('    --model_dir /models/llama-3-8b \\')
print('    --output_dir /models/llama-3-8b-trt-ckpt \\')
print('    --tp_size 1 --dtype float16')
print()
print('Step 2: Build TRT engine')
print('  trtllm-build \\')
print('    --checkpoint_dir /models/llama-3-8b-trt-ckpt \\')
print('    --output_dir /models/llama-3-8b-engine \\')
print('    --gemm_plugin float16 \\')
print('    --max_input_len 2048 --max_output_len 512 \\')
print('    --max_batch_size 8')
print()
print('Step 3: Run inference')
print('  python run.py \\')
print('    --engine_dir /models/llama-3-8b-engine \\')
print('    --input_text "Hello, world!"')


## 2. Speedup Sources Analysis

TRT-LLM speedup comes from multiple compounding sources: kernel selection, operator fusion, and precision.


In [ ]:
speedups = {
    'PyTorch eager FP16':           1.0,
    '+ torch.compile':              1.4,
    '+ FlashAttention':              1.8,
    '+ TRT kernel selection':        2.2,
    '+ TRT operator fusion':         2.8,
    '+ FP8 precision (H100)':        3.8,
    '+ in-flight batching':          5.5,
}

import matplotlib.pyplot as plt
names = list(speedups.keys())
vals = list(speedups.values())
plt.figure(figsize=(10, 5))
plt.barh(names, vals, color=plt.cm.viridis(np.linspace(0.2, 0.9, len(names))))
plt.xlabel('Throughput speedup vs PyTorch eager FP16')
plt.title('TensorRT-LLM: Cumulative Speedup Sources')
for i, v in enumerate(vals):
    plt.text(v+0.02, i, f'{v:.1f}x', va='center')
plt.tight_layout()
plt.savefig('trt_speedup.png', dpi=100); plt.show()


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- TensorRT-LLM compiles LLMs to GPU-specific engines by selecting optimal kernels, fusing operations, and calibrating precision.
- TRT-LLM's kernel selection (tactic profiling) alone gives 1.5-2x speedup over PyTorch eager — it benchmarks multiple GEMM implementations and picks the fastest for each shape on the target GPU..
- Day 49 implementation complete.
